# Étape 2 — Suivi des expérimentations avec MLflow

L’objectif de cette étape est de tracer les expérimentations de modélisation avec MLflow.

Pour chaque modèle, les paramètres, les métriques et le modèle entraîné seront enregistrés afin de comparer les performances dans l’interface MLflow.

Les métriques suivies sont :
- accuracy ;
- precision ;
- recall ;
- F1-score ;
- ROC AUC ;
- matrice de confusion ;
- score métier basé sur le coût des erreurs.

Dans ce projet, les faux négatifs sont particulièrement coûteux : un faux négatif correspond à un client risqué prédit comme non risqué, donc à un crédit potentiellement accordé à un client susceptible de ne pas rembourser.

Les expérimentations de modélisation ont été tracées avec MLflow.

Chaque modèle est entraîné dans un script Python reproductible situé dans `src/models/`. Les scripts utilisent `mlflow.start_run()` pour créer un run MLflow et enregistrer les paramètres, les métriques, les tags et le modèle entraîné.

Les modèles suivis sont :

- `DummyClassifier`, utilisé comme baseline naïve ;

- `LogisticRegression`, avec imputation, scaling et pondération des classes ;

- `LightGBM`, avec gestion du déséquilibre via `scale_pos_weight` ;

- `LightGBM` avec optimisation du seuil de décision.

L’interface MLflow permet ensuite de comparer les runs selon les métriques classiques et selon un score métier.

## Baseline — DummyClassifier

Un premier modèle baseline a été entraîné avec `DummyClassifier(strategy="most_frequent")`.

Ce modèle prédit systématiquement la classe majoritaire, c’est-à-dire `TARGET = 0`, qui correspond aux clients sans défaut.

Résultats obtenus sur le jeu de validation :

| Métrique | Valeur |
|---|---:|
| Accuracy | 0.9192 |
| Precision | 0.0000 |
| Recall | 0.0000 |
| F1-score | 0.0000 |
| ROC AUC | 0.5000 |
| Faux négatifs | 4965 |
| Faux positifs | 0 |
| Coût métier | 49650 |

L’accuracy est élevée car le dataset est fortement déséquilibré : environ 92 % des clients ne sont pas en défaut. Cependant, le modèle ne détecte aucun client risqué. Le recall est donc nul.

Ce baseline montre que l’accuracy seule n’est pas suffisante pour évaluer un modèle de scoring crédit. Il faudra privilégier des métriques comme le recall, l’AUC et le score métier.

Le premier modèle baseline est un DummyClassifier qui prédit systématiquement la classe majoritaire, c’est-à-dire TARGET = 0.

Il obtient une accuracy élevée d’environ 91,9 %, car la majorité des clients du dataset ne sont pas en défaut. Cependant, cette métrique est trompeuse dans un contexte de classes déséquilibrées.

L’accuracy mesure uniquement la proportion globale de bonnes prédictions. Elle ne tient pas compte de la classe que le modèle détecte réellement, ni du coût des erreurs. Dans ce cas, le modèle prédit tous les clients comme non risqués : il obtient donc beaucoup de vrais négatifs, mais il ne détecte aucun client en défaut.

La matrice de confusion confirme ce problème : le modèle produit 4965 faux négatifs et 0 vrai positif. Cela signifie que tous les clients réellement en défaut sont classés comme bons clients.

Dans un projet de scoring crédit, cette erreur est particulièrement grave, car un faux négatif correspond à un crédit potentiellement accordé à un client risqué. C’est pourquoi l’accuracy seule est insuffisante. Il faut aussi suivre le recall, la précision, le F1-score, l’AUC et un score métier qui pénalise davantage les faux négatifs.

## Expérience MLflow directement depuis le notebook

Afin de respecter la consigne, une première expérience MLflow est exécutée directement dans ce notebook avec `mlflow.start_run()`.

Cette expérience entraîne un modèle baseline simple avec `DummyClassifier`. Les modèles plus complets, comme la régression logistique et LightGBM, sont ensuite entraînés depuis des scripts Python reproductibles dans `src/models/`.

In [1]:
from pathlib import Path

import mlflow
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline


# Configuration MLflow
PROJECT_ROOT = Path.cwd().parent
MLFLOW_DB_PATH = PROJECT_ROOT / "mlflow.db"

mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB_PATH}")

EXPERIMENT_NAME = "P6_credit_scoring_baseline"
mlflow.set_experiment(EXPERIMENT_NAME)


# Chargement des données
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "train_modeling.csv"

df = pd.read_csv(DATA_PATH)

TARGET_COLUMN = "TARGET"
ID_COLUMN = "SK_ID_CURR"

y = df[TARGET_COLUMN].astype(int)
X = df.drop(columns=[TARGET_COLUMN])

if ID_COLUMN in X.columns:
    X = X.drop(columns=[ID_COLUMN])


# Split train / validation
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)


# Modèle baseline
model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", DummyClassifier(strategy="most_frequent")),
    ]
)


# Run MLflow directement dans le notebook
with mlflow.start_run(run_name="notebook_dummy_classifier_baseline"):
    mlflow.set_tag("project", "P6_MLOps_credit_scoring")
    mlflow.set_tag("step", "notebook_modeling")
    mlflow.set_tag("model_type", "DummyClassifier")
    mlflow.set_tag("notebook", "03_modelisation_mlflow.ipynb")
    mlflow.set_tag(
        "description",
        "Baseline DummyClassifier entraînée directement depuis le notebook avec mlflow.start_run().",
    )

    mlflow.log_param("model", "DummyClassifier")
    mlflow.log_param("strategy", "most_frequent")
    mlflow.log_param("imputer_strategy", "median")
    mlflow.log_param("validation_size", 0.2)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("fn_cost", 10)
    mlflow.log_param("fp_cost", 1)
    mlflow.log_param("n_rows", X.shape[0])
    mlflow.log_param("n_features", X.shape[1])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_valid)

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_valid)[:, 1]
        roc_auc = roc_auc_score(y_valid, y_proba)
    else:
        roc_auc = 0.5

    tn, fp, fn, tp = confusion_matrix(y_valid, y_pred).ravel()

    business_cost = (10 * fn) + fp
    business_cost_per_client = business_cost / len(y_valid)

    metrics = {
        "accuracy": accuracy_score(y_valid, y_pred),
        "precision": precision_score(y_valid, y_pred, zero_division=0),
        "recall": recall_score(y_valid, y_pred, zero_division=0),
        "f1_score": f1_score(y_valid, y_pred, zero_division=0),
        "roc_auc": roc_auc,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "business_cost": business_cost,
        "business_cost_per_client": business_cost_per_client,
    }

    for metric_name, metric_value in metrics.items():
        mlflow.log_metric(metric_name, metric_value)

    mlflow.sklearn.log_model(
        sk_model=model,
        name="model",
        skops_trusted_types=["numpy.dtype"],
    )

metrics

2026/06/26 10:37:36 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


{'accuracy': 0.9192709180189262,
 'precision': 0.0,
 'recall': 0.0,
 'f1_score': 0.0,
 'roc_auc': 0.5,
 'tn': np.int64(56537),
 'fp': np.int64(0),
 'fn': np.int64(4965),
 'tp': np.int64(0),
 'business_cost': np.int64(49650),
 'business_cost_per_client': np.float64(0.8072908198107379)}

## Industrialisation des autres runs

Après cette première expérimentation directement dans le notebook, les modèles plus complets ont été déplacés dans des scripts Python situés dans `src/models/`.

Cette organisation permet de garder un notebook lisible tout en rendant les entraînements reproductibles depuis le terminal.

Les scripts utilisés sont :
- `train_baseline_mlflow.py`
- `train_logistic_regression_mlflow.py`
- `train_lightgbm_mlflow.py`
- `tune_lightgbm_threshold_mlflow.py`
- `register_best_model_mlflow.py`

## Modèle 2 — LogisticRegression avec class_weight balanced

Un second modèle a été entraîné avec une `LogisticRegression` intégrée dans un pipeline scikit-learn. Le pipeline contient une imputation par médiane, un scaling avec `StandardScaler`, puis le modèle de régression logistique.

Le paramètre `class_weight="balanced"` permet de tenir compte du déséquilibre des classes en donnant plus de poids à la classe minoritaire `TARGET = 1`.

Par rapport au `DummyClassifier`, l’accuracy diminue, passant d’environ 91,9 % à 71,0 %. Cette baisse est normale, car le modèle ne prédit plus systématiquement la classe majoritaire. En revanche, le recall passe de 0 à environ 70,5 %, ce qui signifie que le modèle détecte une grande partie des clients réellement en défaut.

La matrice de confusion montre que le nombre de faux négatifs passe de 4965 à 1465. Le modèle détecte donc 3500 clients en défaut qui étaient totalement ignorés par le baseline.

Le coût métier diminue également, passant de 49650 à 31024. Cela montre que, même avec une accuracy plus faible, la régression logistique est plus pertinente métier que le modèle dummy.

Les expérimentations sont suivies dans MLflow au sein de l’expérience `P6_credit_scoring_baseline`. Chaque run enregistre les paramètres du modèle, les métriques classiques, la matrice de confusion, le score métier ainsi que des tags permettant d’identifier le type de modèle et la version du code utilisée.

La comparaison MLflow montre que le DummyClassifier obtient une accuracy élevée, mais uniquement parce qu’il prédit systématiquement la classe majoritaire. Il ne détecte aucun client en défaut, avec un recall égal à 0 et 4965 faux négatifs.

La LogisticRegression obtient une accuracy plus faible, mais elle détecte une partie importante des clients en défaut. Son recall atteint environ 70,5 %, son AUC atteint environ 0,773 et le nombre de faux négatifs diminue de 4965 à 1465.

D’un point de vue métier, la LogisticRegression est donc plus pertinente que le DummyClassifier, car elle réduit le coût métier de 49650 à 31024.

## Modèle 3 — LightGBM baseline pondéré

Un troisième modèle a été entraîné avec LightGBM. Ce modèle est adapté aux données tabulaires et gère nativement les valeurs manquantes. Contrairement à la régression logistique, il ne nécessite pas de scaling.

Le déséquilibre de classes a été pris en compte avec le paramètre `scale_pos_weight`, calculé à partir du ratio entre les clients non-défaillants et les clients en défaut dans le jeu d’entraînement.

Le modèle obtient une accuracy d’environ 73,8 %, un recall d’environ 69,2 % et une AUC d’environ 0,788. Il détecte 3434 clients en défaut et en rate 1531.

Par rapport à la LogisticRegression, LightGBM obtient une AUC plus élevée et un coût métier plus faible. Le coût métier passe de 31024 pour la LogisticRegression à 29912 pour LightGBM. À ce stade, LightGBM est donc le meilleur modèle provisoire selon le critère métier.

| Modèle | Accuracy | Recall | AUC | FN | FP | Coût métier |
|---|---:|---:|---:|---:|---:|---:|
| DummyClassifier | 0.9193 | 0.0000 | 0.5000 | 4965 | 0 | 49650 |
| LogisticRegression | 0.7099 | 0.7049 | 0.7732 | 1465 | 16374 | 31024 |
| LightGBM | 0.7377 | 0.6916 | 0.7881 | 1531 | 14602 | 29912 |

## Optimisation du seuil de décision

Après l'entraînement du modèle LightGBM, plusieurs seuils de décision ont été testés afin de minimiser le coût métier.

Le seuil par défaut de 0.5 donne un coût métier de 29912, avec 1531 faux négatifs et 14602 faux positifs.

L'optimisation du seuil indique qu'un seuil de 0.55 minimise légèrement le coût métier, avec un coût de 29890. Ce seuil réduit le nombre de faux positifs à 11720, mais augmente le nombre de faux négatifs à 1817. Le recall diminue donc de 0.692 à 0.634.

Le gain métier reste limité, mais cette étape montre l'intérêt de ne pas se limiter au seuil par défaut de 0.5. Dans un problème de scoring crédit, le choix du seuil doit être guidé par le coût métier des erreurs, et non uniquement par l'accuracy.

## Versionnement du meilleur modèle dans MLflow

Après comparaison des différents runs MLflow, le modèle LightGBM avec optimisation du seuil obtient le meilleur coût métier.

Le meilleur seuil identifié est 0.55, avec un coût métier de 29890. Ce modèle a été enregistré dans le Model Registry sous le nom `P6_credit_scoring_lightgbm_champion`.

Une première version du modèle a été créée et l’alias `champion` lui a été attribué. Cela permet d’identifier clairement le meilleur modèle actuel et de préparer son utilisation dans les prochaines étapes du cycle MLOps.

In [2]:
from pathlib import Path

import mlflow
import pandas as pd


# On force MLflow à utiliser la base située à la racine du projet
PROJECT_ROOT = Path.cwd().parent
MLFLOW_DB_PATH = PROJECT_ROOT / "mlflow.db"

mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB_PATH}")

EXPERIMENT_NAME = "P6_credit_scoring_baseline"

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    raise ValueError(
        f"Expérience MLflow introuvable : {EXPERIMENT_NAME}. "
        f"Tracking URI utilisé : {mlflow.get_tracking_uri()}"
    )

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
)

columns_to_display = [
    "tags.mlflow.runName",
    "params.model",
    "params.threshold",
    "metrics.accuracy",
    "metrics.precision",
    "metrics.recall",
    "metrics.f1_score",
    "metrics.roc_auc",
    "metrics.fn",
    "metrics.fp",
    "metrics.business_cost",
    "metrics.business_cost_per_client",
    "metrics.best_threshold",
    "metrics.best_business_cost",
    "metrics.best_business_cost_per_client",
    "metrics.best_fn",
    "metrics.best_fp",
    "metrics.best_recall",
    "metrics.best_precision",
]

available_columns = [
    column for column in columns_to_display
    if column in runs.columns
]

runs_comparison = runs[available_columns].copy()

runs_comparison

,tags.mlflow.runName,params.model,params.threshold,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1_score,metrics.roc_auc,metrics.fn,metrics.fp,metrics.business_cost,metrics.business_cost_per_client,metrics.best_threshold,metrics.best_business_cost,metrics.best_business_cost_per_client,metrics.best_fn,metrics.best_fp,metrics.best_recall,metrics.best_precision
0,notebook_dummy_classifier_baseline,DummyClassifier,None,0.919271,0.000000,0.000000,0.000000,0.500000,4965.0,0.0,49650.0,0.807291,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,notebook_summary_mlflow_tracking,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29890.0,0.486,1817.0,11720.0,0.634000,0.21200
2,lightgbm_threshold_0.90,LightGBM,0.9000000000000002,0.920019,0.685484,0.017120,0.033405,0.788063,4880.0,39.0,48839.0,0.794104,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,lightgbm_threshold_0.85,LightGBM,0.8500000000000003,0.919450,0.506433,0.087210,0.148797,0.788063,4532.0,422.0,45742.0,0.743748,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,lightgbm_threshold_0.80,LightGBM,0.8000000000000003,0.913531,0.419443,0.185096,0.256847,0.788063,4046.0,1272.0,41732.0,0.678547,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,lightgbm_threshold_0.75,LightGBM,0.7500000000000002,0.899385,0.351108,0.290433,0.317901,0.788063,3523.0,2665.0,37895.0,0.616159,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,lightgbm_threshold_0.70,LightGBM,0.7000000000000002,0.877646,0.299875,0.386304,0.337646,0.788063,3047.0,4478.0,34948.0,0.568242,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,lightgbm_threshold_0.65,LightGBM,0.6500000000000001,0.850785,0.265165,0.478953,0.341348,0.788063,2587.0,6590.0,32460.0,0.527788,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,lightgbm_threshold_0.60,LightGBM,0.6000000000000002,0.817713,0.235249,0.558912,0.331126,0.788063,2190.0,9021.0,30921.0,0.502764,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,lightgbm_threshold_0.55,LightGBM,0.5500000000000002,0.779893,0.211730,0.634038,0.317451,0.788063,1817.0,11720.0,29890.0,0.486000,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
runs_comparison.sort_values(
    by=["metrics.business_cost"],
    ascending=True,
    na_position="last",
)

,tags.mlflow.runName,params.model,params.threshold,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1_score,metrics.roc_auc,metrics.fn,metrics.fp,metrics.business_cost,metrics.business_cost_per_client,metrics.best_threshold,metrics.best_business_cost,metrics.best_business_cost_per_client,metrics.best_fn,metrics.best_fp,metrics.best_recall,metrics.best_precision
9,lightgbm_threshold_0.55,LightGBM,0.5500000000000002,0.779893,0.211730,0.634038,0.317451,0.788063,1817.0,11720.0,29890.0,0.486000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,lightgbm_threshold_0.50,LightGBM,0.5000000000000001,0.737683,0.190397,0.691641,0.298596,0.788063,1531.0,14602.0,29912.0,0.486358,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,lightgbm_baseline_weighted,LightGBM,None,0.737683,0.190397,0.691641,0.298596,0.788063,1531.0,14602.0,29912.0,0.486358,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,lightgbm_threshold_0.45,LightGBM,0.4500000000000001,0.688449,0.171936,0.749245,0.279689,0.788063,1245.0,17916.0,30366.0,0.493740,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,lightgbm_threshold_0.60,LightGBM,0.6000000000000002,0.817713,0.235249,0.558912,0.331126,0.788063,2190.0,9021.0,30921.0,0.502764,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22,logistic_regression_balanced,LogisticRegression,None,0.709944,0.176109,0.704935,0.281815,0.773160,1465.0,16374.0,31024.0,0.504439,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,lightgbm_threshold_0.40,LightGBM,0.40000000000000013,0.634630,0.155827,0.798187,0.260749,0.788063,1002.0,21469.0,31489.0,0.512000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,lightgbm_threshold_0.65,LightGBM,0.6500000000000001,0.850785,0.265165,0.478953,0.341348,0.788063,2587.0,6590.0,32460.0,0.527788,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,lightgbm_threshold_0.35,LightGBM,0.3500000000000001,0.574680,0.141998,0.846526,0.243201,0.788063,762.0,25396.0,33016.0,0.536828,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,lightgbm_threshold_0.70,LightGBM,0.7000000000000002,0.877646,0.299875,0.386304,0.337646,0.788063,3047.0,4478.0,34948.0,0.568242,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
runs_comparison[
    runs_comparison["metrics.business_cost"].notna()
].sort_values(
    by="metrics.business_cost",
    ascending=True,
)

,tags.mlflow.runName,params.model,params.threshold,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1_score,metrics.roc_auc,metrics.fn,metrics.fp,metrics.business_cost,metrics.business_cost_per_client,metrics.best_threshold,metrics.best_business_cost,metrics.best_business_cost_per_client,metrics.best_fn,metrics.best_fp,metrics.best_recall,metrics.best_precision
9,lightgbm_threshold_0.55,LightGBM,0.5500000000000002,0.779893,0.211730,0.634038,0.317451,0.788063,1817.0,11720.0,29890.0,0.486000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,lightgbm_threshold_0.50,LightGBM,0.5000000000000001,0.737683,0.190397,0.691641,0.298596,0.788063,1531.0,14602.0,29912.0,0.486358,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,lightgbm_baseline_weighted,LightGBM,None,0.737683,0.190397,0.691641,0.298596,0.788063,1531.0,14602.0,29912.0,0.486358,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,lightgbm_threshold_0.45,LightGBM,0.4500000000000001,0.688449,0.171936,0.749245,0.279689,0.788063,1245.0,17916.0,30366.0,0.493740,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,lightgbm_threshold_0.60,LightGBM,0.6000000000000002,0.817713,0.235249,0.558912,0.331126,0.788063,2190.0,9021.0,30921.0,0.502764,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22,logistic_regression_balanced,LogisticRegression,None,0.709944,0.176109,0.704935,0.281815,0.773160,1465.0,16374.0,31024.0,0.504439,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,lightgbm_threshold_0.40,LightGBM,0.40000000000000013,0.634630,0.155827,0.798187,0.260749,0.788063,1002.0,21469.0,31489.0,0.512000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,lightgbm_threshold_0.65,LightGBM,0.6500000000000001,0.850785,0.265165,0.478953,0.341348,0.788063,2587.0,6590.0,32460.0,0.527788,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,lightgbm_threshold_0.35,LightGBM,0.3500000000000001,0.574680,0.141998,0.846526,0.243201,0.788063,762.0,25396.0,33016.0,0.536828,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,lightgbm_threshold_0.70,LightGBM,0.7000000000000002,0.877646,0.299875,0.386304,0.337646,0.788063,3047.0,4478.0,34948.0,0.568242,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
runs_comparison[
    runs_comparison["metrics.best_business_cost"].notna()
].sort_values(
    by="metrics.best_business_cost",
    ascending=True,
)

,tags.mlflow.runName,params.model,params.threshold,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1_score,metrics.roc_auc,metrics.fn,metrics.fp,metrics.business_cost,metrics.business_cost_per_client,metrics.best_threshold,metrics.best_business_cost,metrics.best_business_cost_per_client,metrics.best_fn,metrics.best_fp,metrics.best_recall,metrics.best_precision
1,notebook_summary_mlflow_tracking,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29890.0,0.486,1817.0,11720.0,0.634000,0.21200
19,lightgbm_threshold_tuning,LightGBM,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.55,29890.0,0.486,1817.0,11720.0,0.634038,0.21173


## Run MLflow depuis le notebook

Afin de respecter la consigne, un run MLflow est également créé directement depuis ce notebook avec `mlflow.start_run()`.

Ce run ne réentraîne pas les modèles. Il sert à documenter la comparaison finale des modèles et à enregistrer les résultats principaux de l'étape de modélisation.

In [6]:
from pathlib import Path

import mlflow


# On force le notebook à utiliser la même base MLflow que les scripts
PROJECT_ROOT = Path.cwd().parent
MLFLOW_DB_PATH = PROJECT_ROOT / "mlflow.db"

mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB_PATH}")

EXPERIMENT_NAME = "P6_credit_scoring_baseline"
mlflow.set_experiment(EXPERIMENT_NAME)


with mlflow.start_run(run_name="notebook_summary_mlflow_tracking"):
    # Tags descriptifs
    mlflow.set_tag("project", "P6_MLOps_credit_scoring")
    mlflow.set_tag("step", "notebook_summary")
    mlflow.set_tag("description", "Synthèse des expérimentations MLflow depuis le notebook")
    mlflow.set_tag("notebook", "03_modelisation_mlflow.ipynb")

    # Paramètres principaux
    mlflow.log_param("best_model", "LightGBM")
    mlflow.log_param("best_threshold", 0.55)
    mlflow.log_param("business_cost_formula", "10 * FN + 1 * FP")
    mlflow.log_param("fn_cost", 10)
    mlflow.log_param("fp_cost", 1)

    # Résultats des modèles classiques
    mlflow.log_metric("dummy_business_cost", 49650)
    mlflow.log_metric("logistic_regression_business_cost", 31024)
    mlflow.log_metric("lightgbm_business_cost_threshold_0_50", 29912)

    # Résultat du meilleur seuil
    mlflow.log_metric("best_business_cost", 29890)
    mlflow.log_metric("best_business_cost_per_client", 0.486)
    mlflow.log_metric("best_recall", 0.634)
    mlflow.log_metric("best_precision", 0.212)
    mlflow.log_metric("best_roc_auc", 0.788)
    mlflow.log_metric("best_fn", 1817)
    mlflow.log_metric("best_fp", 11720)
    mlflow.log_metric("best_tp", 3148)
    mlflow.log_metric("best_tn", 44817)

print("Run MLflow créé depuis le notebook.")

Run MLflow créé depuis le notebook.
